# 05. Universe Selection & Strategy Construction

## 📋 개요
모델의 예측 결과를 바탕으로 **수익률 중심의 투자 후보군(Universe)**을 선정합니다.

## ✨ H3 패치 적용 (2026-02-08)
- **Before**: 노트북에 200줄+ 반복 로직 (정확도/수익성/위험도 평가 루프)
- **After**: `select_investment_universe()` 함수 하나로 전체 흐름 캡슐화
- **패턴**: Facade Pattern (복잡한 비즈니스 로직을 모듈로 추상화)

## 🔄 데이터 흐름
```text
03단계 예측 결과 (predictions.parquet)
    ↓
04단계 미래 예측 (forecasts.parquet)
    ↓
02단계 메타 데이터 (dataset.parquet)
    ↓
[select_investment_universe()] ← ✨ 전체 로직 캡슐화
    ├─ 정확도 평가 (과거)
    ├─ 수익성 평가 (미래)
    ├─ 위험도 평가 (미래)
    ├─ Hard Filter 적용
    └─ Top-K 선정
    ↓
최종 투자 후보 (universe_candidates.parquet / investment_report.csv)
```

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

from src.utils.config import load_config, ProjectPaths
from src.universe.select_universe import select_investment_universe

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로 초기화

In [ ]:
# ==========================================
# 설정 로드 및 경로 초기화 (H2 패턴)
# ==========================================
cfg = load_config()
paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

MODEL_DATE = cfg['universe']['model_date']  # 모델 학습 기준일

print(f"📅 날짜 기준 설정:")
print(f"   - 프로젝트 기준일: {cfg['project']['reference_date']}")
print(f"   - 모델 학습 기준일: {MODEL_DATE}")
print(f"\n📁 경로:")
print(f"   - 과거 예측: {paths.get_predictions_parquet()}")
print(f"   - 미래 예측: {paths.get_forecasts_parquet()}")
print(f"   - Universe 출력: {paths.universe_dir}")

## 2️⃣ 데이터 로드

In [ ]:
print("\n📥 데이터 로드 중...")

# ==========================================
# 1. 과거 예측 결과 (정확도 평가용)
# ==========================================
past_pred_path = paths.get_predictions_parquet()
df_past_pred = pd.read_parquet(past_pred_path)
df_past_pred['date'] = pd.to_datetime(df_past_pred['date'])

print(f"\n[과거 예측]")
print(f"   - 파일: {past_pred_path}")
print(f"   - 행수: {len(df_past_pred):,}")
print(f"   - 기간: {df_past_pred['date'].min()} ~ {df_past_pred['date'].max()}")
print(f"   - 종목 수: {df_past_pred['ticker'].nunique()}")

# ==========================================
# 2. 미래 예측 결과 (수익성 평가용)
# ==========================================
future_pred_path = paths.get_forecasts_parquet()
df_future = pd.read_parquet(future_pred_path)
df_future['date'] = pd.to_datetime(df_future['date'])

print(f"\n[미래 예측]")
print(f"   - 파일: {future_pred_path}")
print(f"   - 행수: {len(df_future):,}")
print(f"   - 기간: {df_future['date'].min()} ~ {df_future['date'].max()}")
print(f"   - 종목 수: {df_future['ticker'].nunique()}")

# ==========================================
# 3. Feature 데이터셋 (리스크 메타 정보)
# ==========================================
dataset_path = paths.get_dataset_parquet()
df_meta = pd.read_parquet(dataset_path)
df_meta['date'] = pd.to_datetime(df_meta['date'])

# 최신 날짜의 메타 정보만 사용
latest_meta_date = df_meta['date'].max()
df_meta_latest = df_meta[df_meta['date'] == latest_meta_date].copy()

print(f"\n[메타 데이터]")
print(f"   - 파일: {dataset_path}")
print(f"   - 기준일: {latest_meta_date}")
print(f"   - 종목 수: {df_meta_latest['ticker'].nunique()}")

print("\n✅ 데이터 로드 완료")

## 3️⃣ Universe 선정 실행 (Facade Pattern)

### ✨ 핵심: 전체 로직을 함수 하나로!

**Before (200줄+)**:
```python
# 정확도 평가 루프 (50줄)
for ticker in df_past['ticker'].unique():
    # rmse, directional_accuracy 계산...

# 수익성 평가 루프 (50줄)
for ticker in df_future['ticker'].unique():
    # find_best_trade...

# 위험도 평가 루프 (50줄)
for ticker in df_future['ticker'].unique():
    # calculate_risk_metrics...

# 통합 및 필터링 (50줄)
df_universe = df_accuracy.merge(...)
df_filtered, stats = apply_hard_filters(...)
```

**After (1줄!)**:
```python
results = select_investment_universe(
    df_past_pred, df_future, df_meta_latest,
    model_date=MODEL_DATE, top_k=100
)
```

In [ ]:
print("\n" + "="*65)
print("3️⃣ Universe 선정 실행 (Facade Pattern)")
print("="*65)

# ==========================================
# ✨ 전체 로직을 함수 하나로 캡슐화!
# ==========================================
results = select_investment_universe(
    df_past_predictions=df_past_pred,
    df_future_forecasts=df_future,
    df_meta=df_meta_latest,
    model_date=MODEL_DATE,
    top_k=100,  # 상위 100개 후보 선정
    verbose=True
)

# ==========================================
# 결과 추출
# ==========================================
df_accuracy = results['accuracy']      # 정확도 평가 결과
df_return = results['returns']         # 수익성 평가 결과
df_risk = results['risk']              # 위험도 평가 결과
df_full_universe = results['full']     # 전체 Universe (필터링 후)
df_candidates = results['candidates']  # Top-K 최종 후보
filter_stats = results['filter_stats'] # 필터링 통계

print("\n✅ Universe 선정 완료")
print(f"   - 최종 후보 종목 수: {len(df_candidates):,}개")

## 4️⃣ 사용자 선택을 위한 상세 리포트 생성

In [ ]:
print("\n" + "="*65)
print("4️⃣ 투자 후보 상세 리포트 생성")
print("="*65)

# ==========================================
# 1. ticker_name_map 로드 (종목명 표시)
# ==========================================
try:
    master_path = paths.get_ticker_master()
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    df_candidates['종목명'] = df_candidates['ticker'].map(ticker_name_map)
    print("✅ ticker_master 로드 완료")
except Exception as e:
    print(f"⚠️  ticker_master 로드 실패: {e}")
    df_candidates['종목명'] = df_candidates['ticker']

# ==========================================
# 2. 리포트용 컬럼 정리 (사용자 의사결정 지원)
# ==========================================
df_candidates_report = df_candidates.copy()

df_candidates_report['순위'] = df_candidates_report['return_rank']
df_candidates_report['종목코드'] = df_candidates_report['ticker']

# 수익성 지표
df_candidates_report['예상일평균수익률(로그)'] = df_candidates_report['daily_log_return'].round(6)
df_candidates_report['예상총수익률(%)'] = df_candidates_report['total_return_pct'].round(2)
df_candidates_report['최적보유기간(일)'] = df_candidates_report['hold_days'].astype(int)

# 정확도 지표
df_candidates_report['방향성정확도(%)'] = (
    df_candidates_report['directional_accuracy'] * 100
).round(2)
df_candidates_report['신뢰도(RMSE역수)'] = (
    df_candidates_report['confidence_rmse']
).round(4)
df_candidates_report['RMSE'] = df_candidates_report['rmse'].round(4)

# 위험 지표
df_candidates_report['리스크점수'] = df_candidates_report['risk_composite_raw'].round(3)
df_candidates_report['변동성'] = df_candidates_report['volatility'].round(4)
df_candidates_report['최대낙폭(%)'] = (df_candidates_report['max_drawdown'] * 100).round(2)

# 매매 정보
df_candidates_report['매수일'] = pd.to_datetime(df_candidates_report['buy_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매도일'] = pd.to_datetime(df_candidates_report['sell_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매수가'] = df_candidates_report['buy_price'].round(0).astype(int)
df_candidates_report['매도가'] = df_candidates_report['sell_price'].round(0).astype(int)

# 메타 정보
df_candidates_report['유동성점수'] = df_candidates_report['liquidity_score'].round(0).astype(int)

# ==========================================
# 3. 최종 리포트 컬럼 순서
# ==========================================
report_cols = [
    '순위', '종목코드', '종목명',
    
    # 수익성 (주요 지표)
    '예상일평균수익률(로그)', '예상총수익률(%)', '최적보유기간(일)',
    
    # 정확도
    '방향성정확도(%)', '신뢰도(RMSE역수)', 'RMSE',
    
    # 위험
    '리스크점수', '변동성', '최대낙폭(%)',
    
    # 매매 정보
    '매수일', '매도일', '매수가', '매도가',
    
    # 메타
    '유동성점수'
]

df_report = df_candidates_report[report_cols]

# ==========================================
# 4. Top 20 미리보기
# ==========================================
print("\n📊 Top 20 종목 미리보기:")

display_cols_short = [
    '순위', '종목명', '예상총수익률(%)', '최적보유기간(일)',
    '방향성정확도(%)', '신뢰도(RMSE역수)', '리스크점수',
    '매수가', '매도가'
]

display(df_report[display_cols_short].head(20))

print("\n✅ 리포트 생성 완료")

## 5️⃣ 결과 저장

In [ ]:
print("\n" + "="*65)
print("5️⃣ 결과 저장")
print("="*65)

# ==========================================
# 1. 전체 Universe (평가 완료)
# ==========================================
full_universe_path = paths.get_universe_full()
df_full_universe.to_parquet(full_universe_path, index=False)
print(f"\n💾 전체 Universe 저장: {full_universe_path}")
print(f"   - 종목 수: {len(df_full_universe):,}")

# ==========================================
# 2. Top-K 후보 (Parquet)
# ==========================================
candidates_path = paths.get_universe_candidates()
df_candidates.to_parquet(candidates_path, index=False)
print(f"\n💾 투자 후보 저장: {candidates_path}")
print(f"   - 종목 수: {len(df_candidates):,}")

# ==========================================
# 3. 상세 리포트 (CSV, 사람 가독성 우선)
# ==========================================
report_csv_path = paths.get_investment_report_csv()
df_report.to_csv(report_csv_path, index=False, encoding='utf-8-sig')
print(f"\n💾 상세 리포트 저장: {report_csv_path}")
print(f"   - 형식: CSV (Excel 호환)")
print(f"   - 컬럼 수: {len(report_cols)}개")

# ==========================================
# 4. Excel용 요약 시트 (선택)
# ==========================================
try:
    excel_path = paths.get_investment_report_excel()
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Top 20 요약
        df_report.head(20).to_excel(writer, sheet_name='Top20', index=False)
        
        # Sheet 2: 전체 후보
        df_report.to_excel(writer, sheet_name='전체후보', index=False)
        
        # Sheet 3: 위험별 분류
        df_risk_groups = df_report.copy()
        df_risk_groups['위험등급'] = pd.cut(
            df_risk_groups['리스크점수'],
            bins=[0, 0.3, 0.5, 0.7, 1.0],
            labels=['낮음', '보통', '높음', '매우높음']
        )
        
        for risk_level in ['낮음', '보통', '높음', '매우높음']:
            df_level = df_risk_groups[df_risk_groups['위험등급'] == risk_level]
            if len(df_level) > 0:
                df_level.to_excel(writer, sheet_name=f'위험_{risk_level}', index=False)
    
    print(f"\n💾 Excel 리포트 저장: {excel_path}")
    print(f"   - Sheet: Top20, 전체후보, 위험_낮음, 위험_보통 등")
    
except Exception as e:
    print(f"\n⚠️  Excel 저장 실패 (openpyxl 필요): {e}")

print("\n" + "="*65)
print("✅ [Step 5] Universe 선정 완료")
print("="*65)
print(f"\n💡 다음 단계:")
print(f"   1. Excel/CSV 파일 열기: {report_csv_path.name}")
print(f"   2. 수익률, 정확도, 위험을 종합 검토")
print(f"   3. 최종 투자 종목 수동 선택 (권장: 20~30개)")
print(f"   4. 선택한 종목으로 06단계 포트폴리오 최적화 진행")

## 📊 전체 후보 요약 통계 (선택)

In [ ]:
print("\n📊 전체 후보 요약 통계:")

# 수익성 분포
print(f"\n[수익률 분포]")
print(f"   - 10% 이상: {(df_report['예상총수익률(%)'] >= 10).sum()}개")
print(f"   - 5~10%: {((df_report['예상총수익률(%)'] >= 5) & (df_report['예상총수익률(%)'] < 10)).sum()}개")
print(f"   - 0~5%: {((df_report['예상총수익률(%)'] >= 0) & (df_report['예상총수익률(%)'] < 5)).sum()}개")
print(f"   - 음수: {(df_report['예상총수익률(%)'] < 0).sum()}개")

# 보유기간 분포
print(f"\n[보유기간 분포]")
print(f"   - 5일 이하: {(df_report['최적보유기간(일)'] <= 5).sum()}개")
print(f"   - 6~10일: {((df_report['최적보유기간(일)'] > 5) & (df_report['최적보유기간(일)'] <= 10)).sum()}개")
print(f"   - 11~20일: {((df_report['최적보유기간(일)'] > 10) & (df_report['최적보유기간(일)'] <= 20)).sum()}개")
print(f"   - 21일 이상: {(df_report['최적보유기간(일)'] > 20).sum()}개")

# 정확도 분포
print(f"\n[정확도 분포]")
print(f"   - 방향성 70% 이상: {(df_report['방향성정확도(%)'] >= 70).sum()}개")
print(f"   - 방향성 60~70%: {((df_report['방향성정확도(%)'] >= 60) & (df_report['방향성정확도(%)'] < 70)).sum()}개")
print(f"   - 방향성 50~60%: {((df_report['방향성정확도(%)'] >= 50) & (df_report['방향성정확도(%)'] < 60)).sum()}개")
print(f"   - 신뢰도 0.7 이상: {(df_report['신뢰도(RMSE역수)'] >= 0.7).sum()}개")

# 위험도 분포
print(f"\n[위험도 분포]")
print(f"   - 낮은 리스크 (0~0.3): {(df_report['리스크점수'] <= 0.3).sum()}개")
print(f"   - 보통 리스크 (0.3~0.5): {((df_report['리스크점수'] > 0.3) & (df_report['리스크점수'] <= 0.5)).sum()}개")
print(f"   - 높은 리스크 (0.5~0.7): {((df_report['리스크점수'] > 0.5) & (df_report['리스크점수'] <= 0.7)).sum()}개")
print(f"   - 매우 높은 리스크 (0.7+): {(df_report['리스크점수'] > 0.7).sum()}개")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
1. **`universe_candidates.parquet`**: 시스템이 선정한 Top-K 후보군 (데이터 처리용)
2. **`investment_report.csv` / `.xlsx`**: 사용자가 직접 보고 판단할 수 있는 **상세 분석 리포트** ⭐

### 🎯 투자 후보 결정 가이드
생성된 엑셀 리포트를 열고 다음 순서로 검토하는 것을 권장합니다.

1. **상위 20위 검토**: 기대 수익률이 가장 높은 종목들
2. **신뢰도 교차 검증**:
   - `방향성정확도(%)`: 최소 60% 이상인가?
   - `신뢰도(RMSE역수)`: 값이 너무 낮지 않은가?
3. **리스크 확인**:
   - `리스크점수`: 0.7 이상이면 매우 위험
   - `최대낙폭(%)`: 감당 가능한 수준인가?
4. **매매 계획**:
   - `최적보유기간`과 `매수/매도 목표가` 참고

### 🚀 다음 작업
- **06단계**: 포트폴리오 최적화 (MVO 등)
- **백테스트**: 과거 구간 시뮬레이션